In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datasets import load_dataset
from collections import Counter
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
!pip install -q datasets

In [4]:
dataset = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")

print(dataset)
print("\nExample:")
print(dataset["train"][0])

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Example:
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [5]:
label_list = dataset["train"].features["ner_tags"].feature.names
print("\nEntity labels:", label_list)


Entity labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [6]:
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

for i in range(3):
    tokens = dataset["train"][i]["tokens"]
    tags = [id2label[t] for t in dataset["train"][i]["ner_tags"]]
    print(f"\nSentence {i}:")
    for tok, tag in zip(tokens, tags):
        print(f"{tok:15} {tag}")

print("\nTrain size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))


Sentence 0:
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O

Sentence 1:
Peter           B-PER
Blackburn       I-PER

Sentence 2:
BRUSSELS        B-LOC
1996-08-22      O

Train size: 14041
Validation size: 3250
Test size: 3453


In [7]:
all_tags = [id2label[t] for ex in dataset["train"]["ner_tags"] for t in ex]
print("\nTag distribution:")
print(Counter(all_tags))


Tag distribution:
Counter({'O': 169578, 'B-LOC': 7140, 'B-PER': 6600, 'B-ORG': 6321, 'I-PER': 4528, 'I-ORG': 3704, 'B-MISC': 3438, 'I-LOC': 1157, 'I-MISC': 1155})


In [8]:
word_counts = Counter()
for tokens in dataset["train"]["tokens"]:
    word_counts.update([t.lower() for t in tokens])

word2idx = {"<PAD>": 0, "<UNK>": 1}
for word in word_counts:
    word2idx[word] = len(word2idx)

idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(word2idx)
print("Vocab size:", vocab_size)

Vocab size: 21011


In [9]:
char_counts = Counter()
for tokens in dataset["train"]["tokens"]:
    for tok in tokens:
        char_counts.update(tok)

char2idx = {"<PAD>": 0, "<UNK>": 1}
for ch in char_counts:
    char2idx[ch] = len(char2idx)

print("Char vocab size:", len(char2idx))

Char vocab size: 86


In [10]:
num_labels = len(label_list)
print("Num labels:", num_labels)

Num labels: 9


In [11]:
!wget -q http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip glove.6B.100d.txt

In [12]:
embedding_dim = 100
embeddings_index = {}
with open("glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vec = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vec

print("Loaded GloVe vectors:", len(embeddings_index))

Loaded GloVe vectors: 400000


In [13]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))
found = 0
for word, i in word2idx.items():
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[i] = vec
        found += 1

print(f"Words found in GloVe: {found}/{vocab_size} ({found/vocab_size:.1%})")

Words found in GloVe: 18415/21011 (87.6%)


In [14]:
def encode_sentence(tokens):
    return [word2idx.get(t.lower(), word2idx["<UNK>"]) for t in tokens]

def encode_labels(tags):
    return list(tags)

MAX_LEN = 128

In [15]:
def pad_sequence(seq, max_len, pad_value=0):
    seq = seq[:max_len]
    return seq + [pad_value] * (max_len - len(seq))

In [16]:
def prepare_split(split):
    X, y, lengths = [], [], []
    for tokens, tags in zip(dataset[split]["tokens"], dataset[split]["ner_tags"]):
        enc_x = encode_sentence(tokens)
        enc_y = encode_labels(tags)
        lengths.append(min(len(enc_x), MAX_LEN))
        X.append(pad_sequence(enc_x, MAX_LEN, pad_value=word2idx["<PAD>"]))
        y.append(pad_sequence(enc_y, MAX_LEN, pad_value=-100)) 
    return np.array(X), np.array(y), np.array(lengths)

In [17]:
X_train, y_train, len_train = prepare_split("train")
X_val, y_val, len_val = prepare_split("validation")
X_test, y_test, len_test = prepare_split("test")

print("\nShapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


Shapes:
X_train: (14041, 128) y_train: (14041, 128)
X_val: (3250, 128) y_val: (3250, 128)
X_test: (3453, 128) y_test: (3453, 128)


In [18]:
class NERDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.tensor(X), torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(NERDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(NERDataset(X_val, y_val), batch_size=32)

In [19]:
class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_labels, emb_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(emb_matrix, dtype=torch.float32))
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_labels)
    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.fc(out)

In [20]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_labels, emb_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(emb_matrix, dtype=torch.float32))
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
    def forward(self, x):
        out, _ = self.lstm(self.embedding(x))
        return self.fc(out)

In [21]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)


In [22]:
def train_model(model, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b).view(-1, num_labels), y_b.view(-1))
            loss.backward(); optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                val_loss += criterion(model(X_b).view(-1, num_labels), y_b.view(-1)).item()

        print(epoch, train_loss/len(train_loader), val_loss/len(val_loader))

In [23]:
def get_predictions(model, X, y):
    model.eval()
    X_t = torch.tensor(X).to(device)
    with torch.no_grad():
        preds = model(X_t).argmax(-1).cpu().numpy()
    true_labels, pred_labels = [], []
    for i in range(len(y)):
        seq_true, seq_pred = [], []
        for j in range(len(y[i])):
            if y[i][j] != -100:
                seq_true.append(id2label[y[i][j]])
                seq_pred.append(id2label[preds[i][j]])
        true_labels.append(seq_true)
        pred_labels.append(seq_pred)
    return true_labels, pred_labels

In [24]:
def get_entities(seq):
    entities = []
    start, etype = None, None
    for i, tag in enumerate(seq + ["O"]):
        if tag.startswith("B-"):
            if start is not None:
                entities.append((etype, start, i))
            start, etype = i, tag[2:]
        elif tag.startswith("I-") and etype == tag[2:]:
            continue
        else:
            if start is not None:
                entities.append((etype, start, i))
            start, etype = None, None
    return entities

def entity_report(y_true, y_pred):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for t_seq, p_seq in zip(y_true, y_pred):
        t_ents, p_ents = set(get_entities(t_seq)), set(get_entities(p_seq))
        for e in p_ents:
            if e in t_ents: tp[e[0]] += 1
            else: fp[e[0]] += 1
        for e in t_ents:
            if e not in p_ents: fn[e[0]] += 1
    for etype in sorted(set(tp) | set(fp) | set(fn)):
        p = tp[etype] / (tp[etype] + fp[etype] + 1e-9)
        r = tp[etype] / (tp[etype] + fn[etype] + 1e-9)
        f1 = 2 * p * r / (p + r + 1e-9)
        print(f"{etype:6} precision={p:.2f} recall={r:.2f} f1={f1:.2f}")

In [25]:
model = LSTMTagger(vocab_size, embedding_dim, 128, num_labels, embedding_matrix).to(device)
train_model(model)
torch.save(model.state_dict(), "lstm_model.pt")

0 0.3940228303567695 0.2222473185655533
1 0.1320498701702625 0.18262252179613592
2 0.08724701692666306 0.16538241445360816
3 0.0639523118840433 0.1597809804073882
4 0.048933001517282124 0.16109593613438455


In [26]:
bilstm_model = BiLSTMTagger(vocab_size, embedding_dim, 128, num_labels, embedding_matrix).to(device)
train_model(bilstm_model)
torch.save(bilstm_model.state_dict(), "bilstm_model.pt")

0 0.3276263983126789 0.17767292374343263
1 0.09876376053568714 0.1442726445658242
2 0.05813894619431202 0.11524148809029629
3 0.03684341473555816 0.1138008565244758
4 0.023791663340454103 0.1324252581278629


In [27]:
y_true, y_pred = get_predictions(bilstm_model, X_test, y_test)
entity_report(y_true, y_pred)

LOC    precision=0.88 recall=0.81 f1=0.85
MISC   precision=0.73 recall=0.67 f1=0.70
ORG    precision=0.72 recall=0.67 f1=0.70
PER    precision=0.88 recall=0.66 f1=0.75


In [28]:
!pip install -q pytorch-crf
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_labels, emb_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(emb_matrix, dtype=torch.float32))
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, x, y=None, mask=None):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        emissions = self.fc(lstm_out)
        if y is not None:
            return -self.crf(emissions, y, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)  

In [29]:
crf_model = BiLSTM_CRF(vocab_size, embedding_dim, 128, num_labels, embedding_matrix).to(device)
optimizer = torch.optim.Adam(crf_model.parameters(), lr=1e-3)

In [30]:
def make_mask(y_batch):
    return (y_batch != -100)

for epoch in range(5):
    crf_model.train()
    train_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        mask = make_mask(y_b)
        y_b_safe = y_b.clone(); y_b_safe[~mask] = 0  
        optimizer.zero_grad()
        loss = crf_model(X_b, y_b_safe, mask=mask)
        loss.backward(); optimizer.step()
        train_loss += loss.item()

    crf_model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            mask = make_mask(y_b)
            y_b_safe = y_b.clone(); y_b_safe[~mask] = 0
            val_loss += crf_model(X_b, y_b_safe, mask=mask).item()

    print(epoch, train_loss/len(train_loader), val_loss/len(val_loader))

torch.save(crf_model.state_dict(), "bilstm_crf_model.pt")

0 4.5468144879797325 2.477326819709703
1 1.3172714006927941 1.9779823990721328
2 0.7437456603541732 1.8655436925736129
3 0.4497260925166971 1.8412443010830413
4 0.293194823051911 1.7997726535095888


In [31]:
def get_predictions_crf(model, X, y):
    model.eval()
    X_t = torch.tensor(X).to(device)
    y_t = torch.tensor(y).to(device)
    mask = (y_t != -100)
    with torch.no_grad():
        preds = model(X_t, mask=mask)  # list of lists, one per sequence, unpadded
    true_labels, pred_labels = [], []
    for i in range(len(y)):
        seq_len = mask[i].sum().item()
        seq_true = [id2label[y[i][j]] for j in range(seq_len)]
        seq_pred = [id2label[p] for p in preds[i]]
        true_labels.append(seq_true)
        pred_labels.append(seq_pred)
    return true_labels, pred_labels

y_true, y_pred = get_predictions_crf(crf_model, X_test, y_test)
entity_report(y_true, y_pred)

LOC    precision=0.89 recall=0.82 f1=0.86
MISC   precision=0.76 recall=0.66 f1=0.70
ORG    precision=0.75 recall=0.70 f1=0.72
PER    precision=0.86 recall=0.69 f1=0.77


In [32]:
!pip install -q transformers

from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)  
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [33]:
tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

bert_model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

data_collator = DataCollatorForTokenClassification(tokenizer)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels, pred_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        seq_true, seq_pred = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                seq_true.append(id2label[l])
                seq_pred.append(id2label[p])
        true_labels.append(seq_true)
        pred_labels.append(seq_pred)

    tp, fp, fn = 0, 0, 0
    for t_seq, p_seq in zip(true_labels, pred_labels):
        t_ents, p_ents = set(get_entities(t_seq)), set(get_entities(p_seq))
        tp += len(t_ents & p_ents)
        fp += len(p_ents - t_ents)
        fn += len(t_ents - p_ents)

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return {"precision": precision, "recall": recall, "f1": f1}

In [35]:
args = TrainingArguments(
    output_dir="distilbert-ner",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=bert_model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,   
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.125994,0.901380,0.901380,0.901380
2,0.354988,0.096995,0.935348,0.932514,0.933929
3,0.090311,0.094597,0.936281,0.937227,0.936754


TrainOutput(global_step=1317, training_loss=0.18441876300646667, metrics={'train_runtime': 177.8734, 'train_samples_per_second': 236.814, 'train_steps_per_second': 7.404, 'total_flos': 564906532257774.0, 'train_loss': 0.18441876300646667, 'epoch': 3.0})

In [36]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

preds_output = trainer.predict(tokenized_dataset["test"])
logits, labels = preds_output.predictions, preds_output.label_ids
preds = np.argmax(logits, axis=-1)

true_labels, pred_labels = [], []
for pred_row, label_row in zip(preds, labels):
    seq_true, seq_pred = [], []
    for p, l in zip(pred_row, label_row):
        if l != -100:
            seq_true.append(id2label[l])
            seq_pred.append(id2label[p])
    true_labels.append(seq_true)
    pred_labels.append(seq_pred)

entity_report(true_labels, pred_labels)

{'eval_loss': 0.1968565285205841, 'eval_precision': 0.8925167666782752, 'eval_recall': 0.8953611898015411, 'eval_f1': 0.8939367150734688, 'eval_runtime': 3.8325, 'eval_samples_per_second': 900.977, 'eval_steps_per_second': 14.09, 'epoch': 3.0}
LOC    precision=0.90 recall=0.92 f1=0.91
MISC   precision=0.78 recall=0.79 f1=0.78
ORG    precision=0.86 recall=0.85 f1=0.86
PER    precision=0.97 recall=0.96 f1=0.96


In [37]:
import gradio as gr
import torch

LABEL_COLORS = {
    "PER": "#ff6b6b", "ORG": "#4ecdc4", "LOC": "#ffd93d", "MISC": "#a78bfa"
}
LABEL_NAMES = {"PER": "Person", "ORG": "Organization", "LOC": "Location", "MISC": "Miscellaneous"}

def predict_entities(text):
    if not text.strip():
        return [], "Enter some text above to see detected entities."

    tokens = text.split()
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    preds = logits.argmax(-1)[0].cpu().numpy()

    word_ids = inputs.word_ids(batch_index=0)
    word_preds = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in word_preds:
            word_preds[word_id] = id2label[preds[idx]]

    highlighted = []
    entity_count = {"PER": 0, "ORG": 0, "LOC": 0, "MISC": 0}
    for i, tok in enumerate(tokens):
        tag = word_preds.get(i, "O")
        etype = tag[2:] if tag != "O" else None
        if etype:
            entity_count[etype] += 1
        highlighted.append((tok, etype))

    summary = "  ".join(f"{LABEL_NAMES[k]}: {v}" for k, v in entity_count.items() if v > 0)
    summary = summary if summary else "No entities detected."
    return highlighted, summary

CUSTOM_CSS = """
.gradio-container { max-width: 900px !important; margin: auto; }
#title { text-align: center; margin-bottom: 4px; }
#subtitle { text-align: center; color: #888; margin-bottom: 24px; }
.legend-badge {
    display: inline-block; padding: 4px 12px; border-radius: 16px;
    font-size: 13px; font-weight: 600; margin: 0 6px 6px 0; color: #1a1a1a;
}
"""

legend_html = "".join(
    f'<span class="legend-badge" style="background:{color}">{LABEL_NAMES[tag]}</span>'
    for tag, color in LABEL_COLORS.items()
)

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="teal")) as demo:
    gr.Markdown("# 🔎 Named Entity Recognition", elem_id="title")
    gr.Markdown("Fine-tuned DistilBERT · trained on CoNLL-2003 · live detection", elem_id="subtitle")
    gr.HTML(f'<div style="text-align:center; margin-bottom:20px">{legend_html}</div>')

    with gr.Row():
        with gr.Column(scale=1):
            text_input = gr.Textbox(
                label="Input text",
                placeholder="e.g. Elon Musk visited Berlin last July with a Tesla team.",
                lines=4,
            )
            gr.Examples(
                examples=[
                    "Barack Obama met Angela Merkel in Berlin on Monday.",
                    "Apple Inc. announced record profits in California.",
                    "The United Nations held a summit in Geneva last March.",
                ],
                inputs=text_input,
            )
        with gr.Column(scale=1):
            output = gr.HighlightedText(label="Detected entities (live)", color_map=LABEL_COLORS)
            summary_box = gr.Textbox(label="Summary", interactive=False)

    # Live highlighting as the user types, debounced so it doesn't fire every keystroke
    text_input.change(
        fn=predict_entities,
        inputs=text_input,
        outputs=[output, summary_box],
        show_progress="hidden",
    ).then(lambda: None)  # no-op, keeps chaining pattern simple if you add more later

demo.launch(share=True)

/tmp/ipykernel_23/3842786095.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="teal")) as demo:
/tmp/ipykernel_23/3842786095.py:53: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(primary_hue="teal")) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://af0e4077c1f074b901.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [38]:
import json, torch

bert_model.save_pretrained("/kaggle/working/distilbert-ner")
tokenizer.save_pretrained("/kaggle/working/distilbert-ner")

with open("/kaggle/working/distilbert-ner/id2label.json", "w") as f:
    json.dump(id2label, f)

torch.save(model.state_dict(), "/kaggle/working/lstm_model.pt")
torch.save(bilstm_model.state_dict(), "/kaggle/working/bilstm_model.pt")
torch.save(crf_model.state_dict(), "/kaggle/working/bilstm_crf_model.pt")

with open("/kaggle/working/word2idx.json", "w") as f:
    json.dump(word2idx, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [39]:
!zip -r /kaggle/working/ner_deployment.zip /kaggle/working/distilbert-ner

  adding: kaggle/working/distilbert-ner/ (stored 0%)
  adding: kaggle/working/distilbert-ner/config.json (deflated 53%)
  adding: kaggle/working/distilbert-ner/tokenizer.json (deflated 71%)
  adding: kaggle/working/distilbert-ner/id2label.json (deflated 47%)
  adding: kaggle/working/distilbert-ner/tokenizer_config.json (deflated 42%)
  adding: kaggle/working/distilbert-ner/model.safetensors (deflated 8%)
